In [1]:
import pandas as pd
import numpy as np

In [2]:
cc_calls = pd.read_csv('../../data/raw/cc_calls.csv', low_memory=False)
cc_calls.columns = cc_calls.columns.str.lower().str.replace(' ', '_')
cc_calls.shape

(32882, 33)

Dropping duplicate rows

In [3]:
cc_calls.duplicated().sum()

np.int64(93)

In [4]:
cc_calls = cc_calls.drop_duplicates()
cc_calls.shape

(32789, 33)

Dropping rows with null co_ref

In [5]:
cc_calls['co_ref'].isnull().sum()

np.int64(1153)

In [6]:
cc_calls = cc_calls.dropna(subset=['co_ref'])
cc_calls.shape

(31636, 33)

Standardizing values in direction column

In [7]:
cc_calls['direction'].value_counts()

direction
OUT_BOUND    24292
IN_BOUND      7344
Name: count, dtype: int64

In [8]:
cc_calls['direction'] = cc_calls['direction'].replace({
    'OUT_BOUND': 'Outbound',
    'IN_BOUND': 'Inbound'
})
cc_calls['direction'].value_counts()

direction
Outbound    24292
Inbound      7344
Name: count, dtype: int64

Converting proper datatype of dates

In [9]:
cc_calls['call_date'].head()

0    08-05-2025
1    25-11-2024
2    23-10-2024
3    13-01-2025
4    19-03-2025
Name: call_date, dtype: str

In [10]:
cc_calls['call_date'] = pd.to_datetime(cc_calls['call_date'], format='%d-%m-%Y')
cc_calls['call_date'].head()

0   2025-05-08
1   2024-11-25
2   2024-10-23
3   2025-01-13
4   2025-03-19
Name: call_date, dtype: datetime64[us]

Dropping irrelevant columns<br>
1- analysed_call -> has only one unique value<br>
2- call_year -> can be derived from call_date

In [11]:
cc_calls['analysed_call'].unique()

array([1])

In [12]:
cc_calls['call_year'].unique()

array([2025, 2024, 2026])

In [13]:
cc_calls = cc_calls.drop(columns=['analysed_call', 'call_year'])
cc_calls.shape

(31636, 31)

Handling null values in cc_calls

In [14]:
cc_calls.isnull().sum()[cc_calls.isnull().sum() > 0]

cc_care_package                             136
cc_care_package_discussed                   136
cc_urgency_getting_on_site                  136
cc_external_consultant                      136
cc_agent_cross_sell_attempt                 136
cc_customer_issues_concerns                 136
cc_business_struggles_financial_hardship    136
cc_call_initiated_by                        136
cc_questionnaire_completion                  32
cc_chasing_response                          32
cc_issues_within_questionnaire              443
cc_login_issues                              33
cc_platform_issues                           33
cc_dissatisfaction_time_to_complete          33
cc_process_complexity_concerns               38
cc_questions_harder_than_expected            37
cc_dissatisfaction_support                   36
cc_contractor_sentiment                      95
cc_contractor_sentiment_start_score          95
cc_contractor_sentiment_end_score            95
cc_contractor_sentiment_overall_score   

In [15]:
print('Total rows with null values:', cc_calls.isnull().any(axis=1).sum())

Total rows with null values: 680


In [16]:
null_rows = cc_calls[cc_calls.isnull().any(axis=1)]
non_null_rows = cc_calls[~cc_calls.isnull().any(axis=1)]
print(f'Rows being dropped: {len(null_rows)} out of {len(cc_calls)} ({len(null_rows)/len(cc_calls)*100:.2f}%)')
print('Direction distribution (dropped vs kept):')
print('Dropped:', null_rows['direction'].value_counts(normalize=True).to_dict())
print('Kept:   ', non_null_rows['direction'].value_counts(normalize=True).to_dict())

Rows being dropped: 680 out of 31636 (2.15%)
Direction distribution (dropped vs kept):
Dropped: {'Outbound': 0.7044117647058824, 'Inbound': 0.29558823529411765}
Kept:    {'Outbound': 0.7692531334797778, 'Inbound': 0.23074686652022225}


In [17]:
cc_calls = cc_calls.dropna()
cc_calls.shape

(30956, 31)

In [18]:
print('Total rows with null values:', cc_calls.isnull().any(axis=1).sum())

Total rows with null values: 0


Saving cleaned dataset

In [19]:
import os
os.makedirs('../../data/cleaned', exist_ok=True)
cc_calls.to_csv('../../data/cleaned/cc_calls_cleaned.csv', index=False)
print('Saved! Shape:', cc_calls.shape)

Saved! Shape: (30956, 31)
